# Phase 2 · Notebook 7 — Predict the susceptibility surface

The trained model has only seen a sample of pixels. Here we apply it to **every** pixel in the area to produce a continuous machine-learning flood-susceptibility map, then compare it against Phase 1's weighted-overlay map and the observed SAR flood.

## Step 1 — Load the model and the feature stack

In [1]:
import numpy as np
import rasterio
import xgboost as xgb
import matplotlib.pyplot as plt
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name in ("phase2", "notebooks") else Path.cwd()
OUT = REPO / "data" / "outputs"
P2 = REPO / "phase2"

FEATURES = ["elevation", "slope", "dist_river", "builtup", "hand", "curvature", "local_relief"]
model = xgb.XGBClassifier()
model.load_model(P2 / "flood_model.json")

with rasterio.open(OUT / "feature_stack.tif") as src:
    bands = list(src.descriptions)
    stack = src.read().astype("float32")          # (n_features, H, W)
    profile = src.profile
# reorder bands to match the model's feature order
order = [bands.index(f) for f in FEATURES]
stack = stack[order]
print("Feature stack:", stack.shape, "| bands:", [bands[i] for i in order])

Feature stack: (7, 3118, 2228) | bands: ['elevation', 'slope', 'dist_river', 'builtup', 'hand', 'curvature', 'local_relief']


## Step 2 — Predict flood probability for every valid pixel

We flatten the stack to one row per pixel, ask the model for the probability of flooding, and fold the answer back into a map.

In [2]:
with rasterio.open(OUT / "flood_mask.tif") as src:
    flood = src.read(1)
H, W = flood.shape
valid = (flood != 255) & np.isfinite(stack).all(axis=0)

Xall = stack.reshape(len(FEATURES), -1).T          # (pixels, features)
vflat = valid.reshape(-1)
proba = model.predict_proba(Xall[vflat])[:, 1]

susc = np.full(H * W, np.nan, dtype="float32")
susc[vflat] = proba
susc = susc.reshape(H, W)

sprof = profile.copy()
sprof.update(count=1, dtype="float32", nodata=np.nan, compress="lzw")
with rasterio.open(OUT / "flood_susceptibility_ml.tif", "w", **sprof) as dst:
    dst.write(susc, 1)
print("Saved flood_susceptibility_ml.tif | range:",
      round(float(np.nanmin(susc)), 2), "to", round(float(np.nanmax(susc)), 2))

Saved flood_susceptibility_ml.tif | range: 0.0 to 1.0


## Step 3 — Classify into zones

In [3]:
zones = np.full((H, W), 255, dtype="uint8")
zones[valid] = 1
zones[susc >= 0.25] = 2
zones[susc >= 0.50] = 3
zones[susc >= 0.75] = 4
zones[~valid] = 255
zprof = profile.copy(); zprof.update(count=1, dtype="uint8", nodata=255, compress="lzw")
with rasterio.open(OUT / "risk_zones_ml.tif", "w", **zprof) as dst:
    dst.write(zones, 1)

with rasterio.open(OUT / "flood_mask.tif") as src:
    tr = src.transform
px_km2 = (abs(tr.a) * 111.32 * np.cos(np.deg2rad(28.64))) * (abs(tr.e) * 111.32)
for v, lab in [(1, "Low"), (2, "Medium"), (3, "High"), (4, "Very High")]:
    print(f"   {lab:10}: {(zones == v).sum() * px_km2:6.1f} km²")

   Low       :  504.5 km²
   Medium    :   22.7 km²
   High      :   27.1 km²
   Very High :   55.3 km²


## Step 4 — Compare: SAR flood vs Phase 1 overlay vs Phase 2 ML

Do the two risk maps agree with each other, and with the real flood? Visually, both should light up the floodplain — but the ML map should resolve finer structure.

In [4]:
with rasterio.open(OUT / "flood_susceptibility.tif") as src:
    overlay = src.read(1)

fig, ax = plt.subplots(1, 3, figsize=(18, 8))
ax[0].imshow(np.ma.masked_where(flood != 1, flood), cmap="Blues")
ax[0].set_title("Observed SAR flood (Jul 2023)")
ax[1].imshow(np.ma.masked_invalid(overlay), cmap="YlOrRd", vmin=0, vmax=1)
ax[1].set_title("Phase 1 — weighted overlay")
im = ax[2].imshow(np.ma.masked_invalid(susc), cmap="YlOrRd", vmin=0, vmax=1)
ax[2].set_title("Phase 2 — XGBoost ML")
for a in ax:
    a.axis("off")
fig.colorbar(im, ax=ax, shrink=0.5, label="flood probability / risk")
plt.savefig(OUT / "compare_overlay_vs_ml.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved compare_overlay_vs_ml.png")

Saved compare_overlay_vs_ml.png


/var/folders/jr/0tppzf_n6rscbz7qzblfgwbc0000gn/T/ipykernel_16557/3186613323.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Recap

We produced `flood_susceptibility_ml.tif` — a per-pixel flood probability from the trained model — and a side-by-side comparison with Phase 1 and the real flood.

**Next — Notebook 8:** combine the susceptibility with population to estimate **people at risk**.